In [ ]:
import sys
import json
import os
from dotenv import load_dotenv
load_dotenv()
# TO CHANGE
BASEDIR = "../../.."
sys.path.insert(0, BASEDIR)

In [ ]:
from pprint import pprint
import joblib

In [ ]:
from src import PersonalAI, PersonalAIConfig
from src.kg_model import KnowledgeGraphModelConfig
from src.db_drivers.vector_driver.embedders import EmbedderModelConfig
from src.agents import AgentDriverConfig, AgentDriver
from src.agents.utils import AgentConnectorConfig

from src.pipelines.qa import QAPipelineConfig
from src.pipelines.qa.kg_reasoning import KnowledgeGraphReasonerConfig
from src.pipelines.qa.query_preprocessing import QueryPreprocessorConfig
from src.pipelines.qa.query_preprocessing.decomposition import QueryDecomposerConfig
from src.pipelines.qa.query_preprocessing.denoising import QueryDenoiserConfig
from src.pipelines.qa.query_preprocessing.enhancing import QueryEnhancerConfig
from src.pipelines.qa.kg_reasoning.medium_reasoner import MediumKGReasonerConfig
from src.pipelines.qa.answers_aggregation import AnswersAggregatorConfig

from src.pipelines.qa.knowledge_retriever import KnowledgeRetrieverConfig
from src.pipelines.qa.knowledge_retriever.traversal_methods import WaterCirclesSearchConfig

from src.db_drivers.kv_driver.configs import DEFAULT_MIXEDKV_CONFIG
from src.db_drivers.kv_driver import KeyValueDriverConfig
from src.utils.logger import LogLevel, Logger
from src.utils import ModuleType

In [ ]:
MODELS_TOKEN = os.getenv('MODELS_TOKEN')

kg_config = KnowledgeGraphModelConfig(
    nodestree_config=None # Модель дерева вершин строиться не будет
)
EMBEDDER_MODEL_PATH = '../../../models/intfloat/multilingual-e5-large' # PATH TO APPROPRIATE EMBEDDER-MODEL
kg_config.embedders_configs['default_embedder'] = EmbedderModelConfig(
    model_name_or_path=EMBEDDER_MODEL_PATH
)
kg_config.agents_configs['default_llmagent'] = AgentDriverConfig(
    name = 'openai',
    agent_config = AgentConnectorConfig(
        gen_strategy={'seed': 42, 'top_p': 10e-16, 'temperature': 0.0, 'frequency_penalty': 0, 'presence_penalty': 0},
        credentials= {'token': MODELS_TOKEN, 'model': 'Qwen/Qwen3.6-35B-A3B', 'base_url': 'https://llm-api.sk.appliedai.ru/v1'}
    )
)

In [ ]:
pai_config = PersonalAIConfig(
    lang='ru',
    log_level=LogLevel.DEBUG,
    kg_model_config=kg_config,
    qa_pipeline_config=QAPipelineConfig(
        preprocessor_config=QueryPreprocessorConfig(
            denoising_config=QueryDenoiserConfig(),
            enhancing_config=None,
            decomposition_config=QueryDecomposerConfig()
        ),
        reasoner_config=KnowledgeGraphReasonerConfig(
            reasoner_name='medium',
            reasoner_config=MediumKGReasonerConfig(
                enable_searchplan_steps_check=True,
                planlimit_exceeded_behaviour='casual_answer'
            )
        ),
        aggregator_config=AnswersAggregatorConfig(
            answers_summ_behaviour='casual_answer'
        )
    )
)

In [ ]:
personalai = PersonalAI(pai_config)

In [ ]:
agent = AgentDriver.select(g_config.agents_configs['default_llmagent'])
agent.open_connection()

In [ ]:
response, stat = agent.generate("Be a helpfull assistant","What is the purpose of life?")
print(response)

In [ ]:
def load_file(filepath: str) -> str:
    with open(filepath, 'r', encoding='utf-8') as fd:
        data = fd.read()
    return data

TEXT_EXAMPLES = [
    f"[FILENAME]: aggregate.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/aggregate.md")}",
    f"[FILENAME]: compare_people.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/compare_people.md")}",
    f"[FILENAME]: errors.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/errors.md")}",
    f"[FILENAME]: filters.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/filters.md")}",
    f"[FILENAME]: getting_started.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/getting_started.md")}",
    f"[FILENAME]: history.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/history.md")}",
    f"[FILENAME]: limits.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/limits.md")}",
    f"[FILENAME]: param_metrics.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/param_metrics.md")}",
    f"[FILENAME]: rows.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/rows.md")}",
    f"[FILENAME]: rules.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/rules.md")}",
    f"[FILENAME]: top_n.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/top_n.md")}",
]
print(len(TEXT_EXAMPLES))

In [ ]:
extracted_triplets = []
for i, example in enumerate(TEXT_EXAMPLES):
    print(f"{i+1}. {example}")
    update_results, _, _ = personalai.update_memory(example)
    tmp_text_id, tmp_extracted_triplets = update_results
    extracted_triplets += tmp_extracted_triplets
    print("text_id: ", tmp_text_id)
    print("extracted triplets: ", len(tmp_extracted_triplets))

In [ ]:
print(json.dumps(personalai.kg_model.count_items(detailed=True),indent=5))
personalai.kg_model.check_consistency()

In [ ]:
personalai.textid_store.count_items()

In [ ]:
print("mem kv_cache info:")
print(json.dumps(personalai.mem_pipeline.get_cache_stat(), indent=5))
print("mem agent tgen info:")
print(json.dumps(personalai.mem_pipeline.get_agent_tgen_stat(), indent=5))

In [ ]:
answer1, rinfo1, trace1 = personalai.answer_question("Сколько витрин (view) находятся в системе и в скольких схемах они распределены?")
pprint(answer1)

In [ ]:
answer1, rinfo1, trace1 = personalai.answer_question("В какой схеме расположено наибольшее количество витрин и сколько их там?")
pprint(answer1)

In [ ]:
answer1, rinfo1, trace1 = personalai.answer_question("Что вернёт сервер, если в теле запроса не будет ни time_dimensions, ни metrics, ни columns?")
pprint(answer1)

In [ ]:
answer1, rinfo1, trace1 = personalai.answer_question("Какой метод позволяет быстро найти готовый рецепт под задачу?")
pprint(answer1)

In [ ]:
answer1, rinfo1, trace1 = personalai.answer_question("Как получить обзор всех доступных доменов и приёмов?")
pprint(answer1)

In [ ]:
personalai.close_connections()
del personalai